# GloVe: Global Vectors for Word Representation

In this notebook, we'll explore **GloVe (Global Vectors)** and understand how it differs from **Word2Vec**. After learning Word2Vec, GloVe provides an alternative approach to creating word embeddings that combines the best of both count-based and prediction-based methods.

## Learning Objectives
- Understand the intuition behind GloVe
- Learn how GloVe differs from Word2Vec
- Explore GloVe vectors hands-on
- Compare Word2Vec and GloVe on the same tasks

---
# 1. Quick Recap: Word2Vec

Before diving into GloVe, let's briefly recall how **Word2Vec** works:

## Word2Vec Approach (Prediction-Based)

Word2Vec learns word vectors by **predicting** words from their context (or vice versa):

1. **Skip-gram**: Given a target word, predict surrounding context words
   - Input: "cat" → Output: "the", "sat", "on", "mat"
   
2. **CBOW (Continuous Bag of Words)**: Given context words, predict the target word
   - Input: "the", "sat", "on", "mat" → Output: "cat"

Word2Vec uses a **sliding window** that moves through the corpus, learning from **local context** one window at a time.

```
Sentence: "The quick brown fox jumps over the lazy dog"
Window size = 2

Window 1: [The, quick] brown [fox, jumps]  → Learn "brown" from neighbors
Window 2: [quick, brown] fox [jumps, over] → Learn "fox" from neighbors
...
```

**Key characteristic**: Word2Vec only sees local context (the window), not global statistics.

---
# 2. The GloVe Approach

## The Key Insight

GloVe (developed at Stanford in 2014) takes a different approach. Instead of learning from local context windows, GloVe learns from **global word co-occurrence statistics**.

### The Co-occurrence Matrix

First, GloVe builds a **co-occurrence matrix** from the entire corpus:

```
              the    cat    sat    on    mat
    the        0      5      3     8      2
    cat        5      0      4     1      3
    sat        3      4      0     2      1
    on         8      1      2     0      4
    mat        2      3      1     4      0
```

Each cell X[i,j] counts how often word i appears near word j in the entire corpus.

### The Ratio Insight

The genius of GloVe is in how it uses these counts. Consider this example from the original paper:

| Probability Ratio | k = solid | k = gas | k = water | k = fashion |
|-------------------|-----------|---------|-----------|-------------|
| P(k \| ice)       | 1.9×10⁻⁴  | 6.6×10⁻⁵| 3.0×10⁻³  | 1.7×10⁻⁵    |
| P(k \| steam)     | 2.2×10⁻⁵  | 7.8×10⁻⁴| 2.2×10⁻³  | 1.8×10⁻⁵    |
| **P(k\|ice) / P(k\|steam)** | **8.9** | **8.5×10⁻²** | **1.36** | **0.96** |

**Interpretation:**
- "solid" appears much more with "ice" than "steam" → ratio >> 1
- "gas" appears much more with "steam" than "ice" → ratio << 1  
- "water" is related to both equally → ratio ≈ 1
- "fashion" is unrelated to both → ratio ≈ 1

**GloVe trains vectors such that the dot product of word vectors approximates the log of their co-occurrence probability ratio.**

---
# 3. Word2Vec vs GloVe: Key Differences

| Aspect | Word2Vec | GloVe |
|--------|----------|-------|
| **Method** | Prediction-based (neural network) | Count-based + matrix factorization |
| **Training** | Sliding window, one example at a time | Global co-occurrence matrix |
| **Context** | Local (within window) | Global (entire corpus statistics) |
| **Objective** | Predict context/target words | Approximate log co-occurrence ratios |
| **Efficiency** | Can train on streaming data | Requires full corpus to build matrix |
| **Memory** | Lower (processes sequentially) | Higher (stores co-occurrence matrix) |

## Conceptual Difference

```
Word2Vec:  "Learn word meaning by predicting neighbors"
           → Like learning a language by reading sentences

GloVe:     "Learn word meaning from global co-occurrence patterns"
           → Like learning from a dictionary of word associations
```

## When Each Shines

- **Word2Vec**: Better for capturing syntactic patterns, smaller corpora
- **GloVe**: Better for capturing semantic relationships, larger corpora

---
# 4. Hands-On: Loading and Exploring GloVe

Let's load pre-trained GloVe vectors and explore them.

In [ ]:
# Install required packages
!pip install gensim numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import os
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

## Download and Load GloVe Vectors

GloVe vectors are available from Stanford NLP:
- **glove.6B**: Trained on Wikipedia 2014 + Gigaword 5 (6B tokens)
- Available in 50d, 100d, 200d, and 300d versions

In [ ]:
# Setup paths
DATA_PATH = './data'
os.makedirs(DATA_PATH, exist_ok=True)

glove_file = os.path.join(DATA_PATH, 'glove.6B.100d.txt')
word2vec_file = os.path.join(DATA_PATH, 'glove.6B.100d.word2vec.txt')

# Download GloVe if not present
if not os.path.exists(glove_file):
    print("Downloading GloVe vectors (this may take a few minutes)...")
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -P {DATA_PATH}
    !unzip -q {DATA_PATH}/glove.6B.zip -d {DATA_PATH}
    print("Download complete!")
else:
    print("GloVe file already exists.")

In [ ]:
# Convert GloVe format to Word2Vec format (for gensim compatibility)
if not os.path.exists(word2vec_file):
    print("Converting GloVe to Word2Vec format...")
    glove2word2vec(glove_file, word2vec_file)
    print("Conversion complete!")

# Load the vectors
print("Loading GloVe vectors...")
glove_model = KeyedVectors.load_word2vec_format(word2vec_file, binary=False)
print(f"Loaded {len(glove_model)} word vectors with {glove_model.vector_size} dimensions")

## Exploring Word Vectors

Let's look at what a GloVe vector looks like:

In [ ]:
# Get the vector for a word
word = 'king'
vector = glove_model[word]

print(f"Vector for '{word}':")
print(f"Shape: {vector.shape}")
print(f"First 10 values: {vector[:10]}")
print(f"\nMin: {vector.min():.4f}, Max: {vector.max():.4f}, Mean: {vector.mean():.4f}")

In [ ]:
# Visualize a word vector
plt.figure(figsize=(14, 3))
plt.bar(range(len(vector)), vector, color='steelblue', alpha=0.7)
plt.title(f'GloVe Vector for "{word}" (100 dimensions)')
plt.xlabel('Dimension')
plt.ylabel('Value')
plt.axhline(y=0, color='red', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

---
# 5. Semantic Relationships in GloVe

GloVe captures rich semantic relationships. Let's explore them.

In [ ]:
# Find similar words
def show_similar_words(word, topn=10):
    """Display words most similar to the given word."""
    print(f"Words most similar to '{word}':")
    print("-" * 40)
    for similar_word, score in glove_model.most_similar(word, topn=topn):
        print(f"  {similar_word:15} {score:.4f}")

show_similar_words('king')

In [ ]:
# Try different words
print("\n" + "="*50)
show_similar_words('computer')

print("\n" + "="*50)
show_similar_words('pizza')

print("\n" + "="*50)
show_similar_words('happy')

## Word Analogies: The Famous "King - Man + Woman = Queen"

One of the most impressive properties of word embeddings is their ability to capture analogies through vector arithmetic.

In [ ]:
def analogy(a, b, c, topn=5):
    """
    Find word d such that: a is to b as c is to d
    Computed as: d = b - a + c
    """
    print(f"\n{a} : {b} :: {c} : ?")
    print("-" * 40)
    
    # most_similar with positive and negative does: positive - negative
    # We want: b - a + c, so positive=[b, c], negative=[a]
    result = glove_model.most_similar(positive=[b, c], negative=[a], topn=topn)
    
    for word, score in result:
        print(f"  {word:15} {score:.4f}")
    
    return result[0][0]

In [ ]:
# Classic example: king - man + woman = ?
analogy('man', 'king', 'woman')

In [ ]:
# More analogies!
print("\n" + "="*50)
print("GEOGRAPHY")
analogy('paris', 'france', 'rome')      # Capital relationships
analogy('paris', 'france', 'tokyo')

print("\n" + "="*50)
print("VERB TENSES")
analogy('walking', 'walked', 'swimming')  # Past tense
analogy('go', 'went', 'come')

print("\n" + "="*50)
print("COMPARATIVES")
analogy('good', 'better', 'bad')         # Comparative forms
analogy('big', 'bigger', 'small')

print("\n" + "="*50)
print("FAMILY RELATIONSHIPS")
analogy('man', 'woman', 'brother')       # Gender relationships
analogy('man', 'woman', 'uncle')

---
# 6. Visualizing Word Relationships

Let's visualize how GloVe organizes words in the vector space.

In [ ]:
def plot_words_2d(words, title="Word Embeddings Visualization", method='pca'):
    """
    Plot words in 2D using dimensionality reduction.
    
    Args:
        words: List of words to plot
        title: Plot title
        method: 'pca' or 'tsne'
    """
    # Get vectors for words that exist in vocabulary
    valid_words = [w for w in words if w in glove_model]
    vectors = np.array([glove_model[w] for w in valid_words])
    
    # Reduce to 2D
    if method == 'pca':
        reducer = PCA(n_components=2)
    else:
        reducer = TSNE(n_components=2, random_state=42, perplexity=min(5, len(valid_words)-1))
    
    coords = reducer.fit_transform(vectors)
    
    # Plot
    plt.figure(figsize=(12, 8))
    plt.scatter(coords[:, 0], coords[:, 1], c='steelblue', s=100, alpha=0.6)
    
    for i, word in enumerate(valid_words):
        plt.annotate(word, (coords[i, 0], coords[i, 1]), 
                    fontsize=12, ha='center', va='bottom')
    
    plt.title(title, fontsize=14)
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize royalty and gender relationships
royalty_words = ['king', 'queen', 'prince', 'princess', 'man', 'woman', 'boy', 'girl']
plot_words_2d(royalty_words, "Royalty and Gender Words in GloVe Space")

In [ ]:
# Visualize countries and capitals
geo_words = ['france', 'paris', 'germany', 'berlin', 'italy', 'rome', 
             'spain', 'madrid', 'japan', 'tokyo', 'china', 'beijing']
plot_words_2d(geo_words, "Countries and Capitals in GloVe Space")

In [ ]:
# Visualize different semantic categories
category_words = [
    # Animals
    'dog', 'cat', 'horse', 'lion',
    # Fruits
    'apple', 'banana', 'orange', 'grape',
    # Colors
    'red', 'blue', 'green', 'yellow',
    # Professions
    'doctor', 'teacher', 'engineer', 'lawyer'
]
plot_words_2d(category_words, "Different Semantic Categories in GloVe Space")

---
# 7. Understanding the Math Behind GloVe

## The GloVe Objective Function

GloVe minimizes the following objective:

$$J = \sum_{i,j=1}^{V} f(X_{ij}) \left( w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2$$

Where:
- $X_{ij}$ = co-occurrence count of words i and j
- $w_i$ = word vector for word i
- $\tilde{w}_j$ = context vector for word j
- $b_i, \tilde{b}_j$ = bias terms
- $f(X_{ij})$ = weighting function (reduces impact of very frequent co-occurrences)

## The Weighting Function

$$f(x) = \begin{cases} (x/x_{max})^{0.75} & \text{if } x < x_{max} \\ 1 & \text{otherwise} \end{cases}$$

This prevents very common word pairs (like "the" + "is") from dominating the training.

In [ ]:
# Visualize the weighting function
def glove_weighting(x, x_max=100, alpha=0.75):
    """GloVe weighting function."""
    return np.where(x < x_max, (x / x_max) ** alpha, 1.0)

x = np.linspace(0, 150, 300)
y = glove_weighting(x)

plt.figure(figsize=(10, 5))
plt.plot(x, y, 'b-', linewidth=2)
plt.axvline(x=100, color='red', linestyle='--', label='x_max = 100')
plt.xlabel('Co-occurrence count (X_ij)', fontsize=12)
plt.ylabel('Weight f(X_ij)', fontsize=12)
plt.title('GloVe Weighting Function', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("The weighting function:")
print("- Gives less weight to very rare co-occurrences (left side)")
print("- Caps the weight for very frequent co-occurrences (right of x_max)")
print("- This balances the influence of common vs rare word pairs")

---
# 8. Measuring Word Similarity

Both Word2Vec and GloVe use **cosine similarity** to measure how similar two words are.

In [ ]:
def cosine_similarity(v1, v2):
    """Calculate cosine similarity between two vectors."""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Compare some word pairs
word_pairs = [
    ('king', 'queen'),
    ('king', 'man'),
    ('cat', 'dog'),
    ('cat', 'car'),
    ('happy', 'joyful'),
    ('happy', 'sad'),
]

print("Cosine Similarity between word pairs:")
print("=" * 45)
for w1, w2 in word_pairs:
    sim = glove_model.similarity(w1, w2)
    print(f"  {w1:10} - {w2:10} : {sim:.4f}")

In [ ]:
# Visualize similarity as a heatmap
words_to_compare = ['king', 'queen', 'man', 'woman', 'prince', 'princess']

# Build similarity matrix
n = len(words_to_compare)
sim_matrix = np.zeros((n, n))

for i, w1 in enumerate(words_to_compare):
    for j, w2 in enumerate(words_to_compare):
        sim_matrix[i, j] = glove_model.similarity(w1, w2)

# Plot heatmap
plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='Cosine Similarity')
plt.xticks(range(n), words_to_compare, rotation=45, ha='right')
plt.yticks(range(n), words_to_compare)
plt.title('Word Similarity Matrix (GloVe)', fontsize=14)

# Add values to cells
for i in range(n):
    for j in range(n):
        plt.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
# 9. Building a Simple Co-occurrence Matrix

Let's build a small co-occurrence matrix to understand GloVe's foundation.

In [ ]:
from collections import defaultdict

def build_cooccurrence_matrix(sentences, window_size=2):
    """
    Build a co-occurrence matrix from sentences.
    
    Args:
        sentences: List of sentences (each sentence is a list of words)
        window_size: Context window size on each side
    
    Returns:
        cooccurrence: Dictionary of dictionaries with counts
        vocab: Set of all words
    """
    cooccurrence = defaultdict(lambda: defaultdict(int))
    vocab = set()
    
    for sentence in sentences:
        vocab.update(sentence)
        for i, word in enumerate(sentence):
            # Look at context window
            start = max(0, i - window_size)
            end = min(len(sentence), i + window_size + 1)
            
            for j in range(start, end):
                if i != j:
                    context_word = sentence[j]
                    cooccurrence[word][context_word] += 1
    
    return cooccurrence, vocab

In [ ]:
# Example sentences
sentences = [
    ['the', 'cat', 'sat', 'on', 'the', 'mat'],
    ['the', 'dog', 'sat', 'on', 'the', 'floor'],
    ['the', 'cat', 'chased', 'the', 'dog'],
    ['the', 'dog', 'chased', 'the', 'cat'],
    ['a', 'cat', 'is', 'a', 'pet'],
    ['a', 'dog', 'is', 'a', 'pet'],
]

cooccurrence, vocab = build_cooccurrence_matrix(sentences, window_size=2)
vocab = sorted(vocab)

print(f"Vocabulary: {vocab}")
print(f"\nCo-occurrences for 'cat':")
for word, count in sorted(cooccurrence['cat'].items(), key=lambda x: -x[1]):
    print(f"  cat - {word}: {count}")

In [ ]:
# Visualize the co-occurrence matrix
import pandas as pd

# Convert to matrix
matrix = np.zeros((len(vocab), len(vocab)))
for i, w1 in enumerate(vocab):
    for j, w2 in enumerate(vocab):
        matrix[i, j] = cooccurrence[w1][w2]

# Create DataFrame for nice display
df = pd.DataFrame(matrix, index=vocab, columns=vocab).astype(int)

print("Co-occurrence Matrix:")
print(df)

In [ ]:
# Visualize as heatmap
plt.figure(figsize=(10, 8))
plt.imshow(matrix, cmap='Blues')
plt.colorbar(label='Co-occurrence Count')
plt.xticks(range(len(vocab)), vocab, rotation=45, ha='right')
plt.yticks(range(len(vocab)), vocab)
plt.title('Co-occurrence Matrix (Window Size = 2)', fontsize=14)

# Add values
for i in range(len(vocab)):
    for j in range(len(vocab)):
        if matrix[i, j] > 0:
            plt.text(j, i, int(matrix[i, j]), ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- 'cat' and 'dog' have similar co-occurrence patterns")
print("- 'the' co-occurs with many words (common word)")
print("- 'pet' co-occurs with both 'cat' and 'dog'")

---
# 10. Summary: Word2Vec vs GloVe

## Key Takeaways

### Word2Vec
- **Prediction-based**: Learns by predicting words from context
- **Local context**: Only sees words within a sliding window
- **Online learning**: Can train incrementally on new data
- **Two variants**: Skip-gram and CBOW

### GloVe
- **Count-based + factorization**: Learns from global co-occurrence statistics
- **Global context**: Uses entire corpus statistics at once
- **Batch learning**: Requires full corpus to build matrix first
- **Interpretable objective**: Based on word co-occurrence ratios

### In Practice
Both produce high-quality word vectors! The choice often depends on:
- **Corpus size**: GloVe may be better for larger corpora
- **Training flexibility**: Word2Vec is better for streaming/incremental training
- **Available resources**: Use pre-trained vectors when possible!

### Available Pre-trained Vectors
- **GloVe**: glove.6B (Wikipedia), glove.840B (Common Crawl)
- **Word2Vec**: Google News vectors (3M words, 300d)
- **FastText**: Extends Word2Vec with subword information

In [ ]:
# Final demonstration: word relationships
print("="*60)
print("GloVe captures rich semantic relationships!")
print("="*60)

examples = [
    ("Capitals", "france", "paris", "germany"),
    ("Gender", "man", "king", "woman"),
    ("Plurals", "car", "cars", "apple"),
    ("Tense", "walk", "walked", "run"),
]

for category, a, b, c in examples:
    result = glove_model.most_similar(positive=[b, c], negative=[a], topn=1)
    print(f"\n{category}:")
    print(f"  {a} : {b} :: {c} : {result[0][0]} (similarity: {result[0][1]:.3f})")

---
# Exercises

1. **Explore More Analogies**: Find 5 more analogy relationships that work well with GloVe

2. **Compare Similarity Scores**: Choose 10 word pairs and compare their similarity scores. Do the scores match your intuition?

3. **Visualize Your Own Categories**: Create a visualization with words from a domain you're interested in (sports, music, science, etc.)

4. **Find Failures**: Can you find analogies or similarities that don't work well? What might explain the failures?

5. **Different Dimensions**: Load GloVe vectors with different dimensions (50d, 200d, 300d) and compare results. Does dimension size matter?

---
## References

- Pennington, J., Socher, R., & Manning, C. D. (2014). **GloVe: Global Vectors for Word Representation**. EMNLP.
- Mikolov, T., et al. (2013). **Efficient Estimation of Word Representations in Vector Space**. arXiv. (Word2Vec)
- Stanford NLP GloVe: https://nlp.stanford.edu/projects/glove/
- Gensim Documentation: https://radimrehurek.com/gensim/